# PAS-LGT — Phase 1 on Colab

Baseline Seq2Seq Transformer for long-term highway trajectory prediction.

**Before running:** switch the runtime to a GPU
(`Runtime -> Change runtime type -> T4 GPU`). Phase 1 also runs on CPU, just
slower.

The project lives in the `bunes/` subfolder of the shared course repo, which is
public -- so cloning needs no token or SSH key.

Nothing needs to be uploaded: the dataset is generated synthetically inside the
repo. Real NGSIM/highD data is handled in the last section.

## 1. Get the code

In [ ]:
# HTTPS (not the git@ SSH remote) so Colab needs no key -- the repo is public.
REPO_URL = "https://github.com/sepanta-ghonoodi/Distributed-Computing.git"
BRANCH   = "main"
SUBDIR   = "bunes"   # the PAS-LGT project inside the shared course repo

import os, subprocess, sys
from pathlib import Path

REPO_DIR    = Path("/content") / Path(REPO_URL).stem
PROJECT_DIR = REPO_DIR / SUBDIR

def sh(cmd, cwd=None):
    """Run a shell command, streaming output, and fail loudly."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

if REPO_DIR.exists():
    # Re-running after new commits are pushed: fast-forward only, so a dirty or
    # diverged checkout fails visibly instead of silently merging. This is a
    # shared repo, so a teammate's push can legitimately cause that.
    sh(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only", cwd=REPO_DIR)
else:
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")

assert PROJECT_DIR.exists(), (
    f"{PROJECT_DIR} not found -- has the '{SUBDIR}/' folder been pushed yet?"
)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("\nworking directory:", os.getcwd())
sh("git log --oneline -1")

## 2. Environment check

In [ ]:
# Colab already ships torch, numpy, pandas, scipy, pyarrow, yaml, tqdm,
# matplotlib and pytest -- Phase 1 needs no installs at all. This cell only
# reports what is present so a version surprise shows up here rather than 40
# minutes into a training run.
import importlib

import torch

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device: {torch.cuda.get_device_name(0)}")
else:
    print("no GPU attached -- training falls back to CPU (Runtime > Change runtime type)")

for mod in ["numpy", "pandas", "scipy", "pyarrow", "yaml", "tqdm", "matplotlib", "pytest"]:
    try:
        m = importlib.import_module(mod)
        print(f"  OK   {mod:<12} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"  MISS {mod}   <-- run: !pip install {mod}")

In [ ]:
# Only needed from Phase 2 onwards (Link Projection / lane graph / VEC solver).
# Safe to run now; skip it while you are only on Phase 1.
# !pip install -q shapely scikit-learn networkx einops pulp

## 3. Smoke tests

Run these first. They check the agent-frame geometry, the window shapes, and the
teacher-forcing vs. autoregressive-rollout parity -- the invariant that otherwise
fails silently and produces a model that trains fine but predicts garbage.

In [ ]:
!python -m pytest tests/ -q

## 4. Build the dataset

Synthetic multi-lane highway traffic with IDM car-following and lane changes,
resampled to 2 Hz. Roughly 15 seconds to generate.

In [ ]:
!python scripts/make_dummy_data.py --vehicles 200 --lanes 4 --duration 900 --target-hz 2.0

In [ ]:
# Peek at what was produced.
import pandas as pd

df = pd.read_parquet("data/processed/unified.parquet")
print(df.shape, "|", df["vehicle_id"].nunique(), "vehicles")
display(df.head())
print(df[["speed", "accel", "lane_offset", "gap"]].describe().round(2))

## 5. Train

About 8-15 minutes on a T4 for 60 epochs at this size. The constant-velocity
baseline is printed first -- the model must beat it, or something is wrong.

In [ ]:
!python -m src.train --config configs/phase1_colab.yaml

## 6. Evaluate

Autoregressive rollout on the held-out test vehicles, next to the
constant-velocity reference, with the longitudinal/lateral error split.

In [ ]:
!python -m src.evaluate --ckpt runs/phase1_colab/best.pt --split test --plot

In [ ]:
from IPython.display import Image, display

display(Image("runs/phase1_colab/examples_test.png"))

In [ ]:
# Training curve.
import json

import matplotlib.pyplot as plt

hist = json.load(open("runs/phase1_colab/history.json"))
ep = [h["epoch"] for h in hist]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, [h["train_loss"] for h in hist])
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("train loss"); ax[0].set_yscale("log")
ax[0].grid(alpha=0.3)

ax[1].plot(ep, [h.get("val_ade") for h in hist], label="val ADE")
ax[1].plot(ep, [h.get("val_fde") for h in hist], label="val FDE")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("metres"); ax[1].legend(); ax[1].grid(alpha=0.3)
fig.tight_layout()

## 7. Keeping results

Colab wipes `/content` when the runtime disconnects. To keep a checkpoint, copy
it to Drive:

```python
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pas-lgt-runs && cp -r runs/phase1_colab /content/drive/MyDrive/pas-lgt-runs/
```

## 8. Real data (NGSIM / highD)

Upload the raw files to Drive once, then point the preprocessor at them. This
writes the same `unified.parquet`, so every cell above works unchanged.

```python
from google.colab import drive
drive.mount('/content/drive')

# NGSIM: a single trajectory CSV
!python -m src.data.preprocess --source ngsim \
    --raw /content/drive/MyDrive/data/ngsim_us101.csv --target-hz 2.0

# highD: the directory containing *_tracks.csv and *_tracksMeta.csv
!python -m src.data.preprocess --source highd \
    --raw /content/drive/MyDrive/data/highd --target-hz 2.0
```

Then re-run training with `--no-cache` so the window cache is rebuilt from the
new parquet rather than reused from the synthetic run:

```python
!python -m src.train --config configs/phase1_colab.yaml --no-cache
```

## 9. Picking up new commits

After a new commit is pushed, re-run cell 1 -- it fast-forwards the checkout --
then re-run whichever cells you need. No re-upload, no re-setup.

Because this is the shared course repo, `git pull --ff-only` will refuse rather
than merge if the branch has diverged. That is deliberate: it surfaces the
conflict here instead of quietly training on a mixed tree.